# 04 - Model Training

Train the SegFormer-PINN hybrid model:
- Load preprocessed data
- Initialize model
- Train with physics-informed loss
- Evaluate performance
- Save checkpoints

In [ ]:
import sys
sys.path.append('..')

import torch
import yaml
from pathlib import Path
from torch.utils.data import DataLoader

from src.models.segformer_pinn import SegFormerPINN
from src.models.training import UrbanHeatDataset, Trainer, create_model
from src.models.physics_loss import PhysicsInformedLoss

In [ ]:
# Load config
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Create datasets
train_dataset = UrbanHeatDataset(
    data_dir='../data/processed/patches',
    split='train',
    augment=True,
)

val_dataset = UrbanHeatDataset(
    data_dir='../data/processed/patches',
    split='val',
    augment=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=config['training']['num_workers'],
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=config['training']['num_workers'],
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# Create model
model = create_model(config['model'])
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    config=config['training'],
    device=device,
    output_dir='../outputs/checkpoints',
)

# Train
trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=config['training']['num_epochs'],
)

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(trainer.metrics['train_loss'], label='Train')
axes[0].plot(trainer.metrics['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()

axes[1].plot(trainer.metrics['val_rmse'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE (°C)')
axes[1].set_title('Validation RMSE')

axes[2].plot(trainer.metrics['val_r2'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('R²')
axes[2].set_title('Validation R²')

plt.tight_layout()
plt.savefig('../outputs/figures/training_curves.png', dpi=300)
plt.show()

In [ ]:
# Load best model and evaluate
trainer.load_checkpoint('best_model.pth')

test_dataset = UrbanHeatDataset(
    data_dir='../data/processed/patches',
    split='test',
    augment=False,
)

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

test_metrics = trainer.validate(test_loader)
print(f"Test RMSE: {test_metrics['rmse']:.2f}°C")
print(f"Test R²: {test_metrics['r2']:.4f}")